## BERT模型提取嵌入
使用用Transformers库从预训练的BERT模型中提取嵌入。   
预训练模型我们这里使用，不区分大小写的`bert-base-uncased`。`BERT-base`有12个编码器，特征向量大小是`768`。

In [1]:
import torch
from transformers import BertModel, BertTokenizer

torch.set_printoptions(edgeitems=2)
print("")

### 1. 实例化模型和词元分析器

In [2]:
# 下载并加载预训练模型: 第一次调用的时候会下载模型，需要点时间
model = BertModel.from_pretrained("bert-base-uncased")

In [3]:
# 词元分析器
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

### 2. 获取句子的标记和注意力掩码

#### 2.1 标记
我们需要长度是15的标记，长度不够用`[PAD]`来凑。

In [4]:
# 准备句子
text = "I love Python and Pytorch"

In [5]:
# 对句子进行分词，获得标记(Token)
tokens = tokenizer.tokenize(text)

In [6]:
tokens

['i', 'love', 'python', 'and', 'p', '##yt', '##or', '##ch']

In [7]:
# 加2个标记：`[CLS]`: 开头、`[SEP]`:结尾
tokens.insert(0, '[CLS]')
tokens.append('[SEP]')

In [8]:
tokens

['[CLS]', 'i', 'love', 'python', 'and', 'p', '##yt', '##or', '##ch', '[SEP]']

In [9]:
len(tokens)

10

假设我们需要标记长度为15，那么我们再加5个`[PAD]`标记

In [10]:
while len(tokens) < 15:
    tokens.append('[PAD]')
print(tokens)

['[CLS]', 'i', 'love', 'python', 'and', 'p', '##yt', '##or', '##ch', '[SEP]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]']


**我们获取标记的数字ID：**

In [11]:
token_ids = tokenizer.convert_tokens_to_ids(tokens)
print(token_ids)

[101, 1045, 2293, 18750, 1998, 1052, 22123, 2953, 2818, 102, 0, 0, 0, 0, 0]


#### 2.2 注意力掩码
我们来创建个注意力掩码，如果标记是`[PAD]`，注意力掩码的值设置为0，其它的都设置为1.

In [12]:
attention_mask = [0 if token == '[PAD]' else 1 for token in tokens]
print(attention_mask)

[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0]


#### 2.3 得到标记和注意力掩码的张量

In [13]:
token_ids = torch.tensor(token_ids).unsqueeze(0)
attention_mask = torch.tensor(attention_mask).unsqueeze(0)

In [14]:
token_ids.shape

torch.Size([1, 15])

In [15]:
attention_mask.shape

torch.Size([1, 15])

### 3. 获取嵌入向量
我们现在已经有了`token_ids`和`attention_mask`，把它们输入到`BERT`模型中，得到嵌入向量。

In [16]:
results = model(token_ids, attention_mask=attention_mask)

In [17]:
type(results)

transformers.modeling_outputs.BaseModelOutputWithPoolingAndCrossAttentions

In [18]:
len(results), results.keys()

(2, odict_keys(['last_hidden_state', 'pooler_output']))

In [19]:
type(results[0]), type(results[1])

(torch.Tensor, torch.Tensor)

把标记的id和注意力掩码传递给模型，得到嵌入向量。   
返回的输出是一个有2个值的张量对象。
- 第一个值是`hidden_rep`表示隐藏状态的特征，其包括从顶层编码器(`BERT-base`是编码器12)获得的所有标记的特征
- 第二个值是：`cls_head`表示`[CLS]`标记的特征。（也就是整个句子的特征）

In [20]:
results[0].shape

torch.Size([1, 15, 768])

`[1, 15, 768]`表示`[batch_size, sequence_length, hidden_size]`。
- 也就是批量大小值为1，一个句子
- 序列长度等于标记的长度，这里是15
- 隐藏层的大小等于特征向量（嵌入向量）的大小（`BERT-base`模型中是768）

In [21]:
results[0]

tensor([[[-0.0259,  0.3494,  ...,  0.5157,  0.6266],
         [ 0.3756,  0.6662,  ...,  0.7437,  0.2713],
         ...,
         [ 0.0684, -0.0136,  ...,  0.3009, -0.1162],
         [ 0.1553,  0.0536,  ...,  0.2675,  0.0215]]],
       grad_fn=<NativeLayerNormBackward0>)

In [22]:
print(tokens)

['[CLS]', 'i', 'love', 'python', 'and', 'p', '##yt', '##or', '##ch', '[SEP]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]']


In [23]:
results[1].shape

torch.Size([1, 768])

`[1, 768]`表示`[batch_size, hidden_size]`。

`results[1]`具有句子的总特征，我们可以用`results[1]`作为句子`I love Python and Pytorch`的总特征。

In [24]:
# results[1]

我们知道如何从预训练模型`BERT`模型的顶层编码器提取嵌入。   
接下来我们尝试从所有嵌入层获得嵌入。

### 4. 所有编码器层的嵌入
我们使用`BERT-base`它有12层编码器。在下载预训练的BERT模型时，需要设置一个参数`output-hidden_states=True`，将会允许我们从所有编码器层获得嵌入。

In [25]:
# 下载预训练模型，注意参数：output-hidden_states
model2 = BertModel.from_pretrained("bert-base-uncased", output_hidden_states=True)

In [26]:
# tokenizer我们还是使用前面的词元分析器
print(text)
print(tokens)
print(attention_mask)

I love Python and Pytorch
['[CLS]', 'i', 'love', 'python', 'and', 'p', '##yt', '##or', '##ch', '[SEP]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]']
tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0]])


In [27]:
token_ids

tensor([[  101,  1045,  2293, 18750,  1998,  1052, 22123,  2953,  2818,   102,
             0,     0,     0,     0,     0]])

In [28]:
# 上面是我们通过手动添加了`[CLS]`、`[SEP]`、`[PAD]`，我其实也可以直接调用`tokenizer`传递`padding='max_length'即可
tokenizer([text], padding='max_length', max_length=15)

{'input_ids': [[101, 1045, 2293, 18750, 1998, 1052, 22123, 2953, 2818, 102, 0, 0, 0, 0, 0]], 'token_type_ids': [[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]], 'attention_mask': [[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0]]}

In [29]:
# 第一个参数是可以传递列表，也可以直接是一个句子的
tokenizer(text, padding='max_length', max_length=15)

{'input_ids': [101, 1045, 2293, 18750, 1998, 1052, 22123, 2953, 2818, 102, 0, 0, 0, 0, 0], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0]}

**获取嵌入**

In [30]:
results2 = model2(token_ids, attention_mask=attention_mask)
len(results2)

3

**注意**： 现在把`token_ids`和`attention_mask`传给模型后，返回的结果长度是3了。
- 第1个：包含最后编码器中获得的所有标记的特征
- 第2个：表示来自最后的编码器`[CLS]`标记的特征
- 第3个：包含所有编码器层获得的所有标记的特征

In [31]:
results2.keys()

odict_keys(['last_hidden_state', 'pooler_output', 'hidden_states'])

In [32]:
# 分别用3个变量来存储返回的结果
last_hidden_state, pooler_output, hidden_states = results2[0], results2[1], results2[2]

In [33]:
# 查看hidden_states的类型，长度，以及第一个元素的形状
type(hidden_states), len(hidden_states), hidden_states[0].shape

(tuple, 13, torch.Size([1, 15, 768]))

In [34]:
for hidden_i in hidden_states:
    print(hidden_i.shape)

torch.Size([1, 15, 768])
torch.Size([1, 15, 768])
torch.Size([1, 15, 768])
torch.Size([1, 15, 768])
torch.Size([1, 15, 768])
torch.Size([1, 15, 768])
torch.Size([1, 15, 768])
torch.Size([1, 15, 768])
torch.Size([1, 15, 768])
torch.Size([1, 15, 768])
torch.Size([1, 15, 768])
torch.Size([1, 15, 768])
torch.Size([1, 15, 768])


可以发现`hidden_states`是包含从所有编码器层获得的所有标记特征。长度是13，含有从输入嵌入层到最后的编码器层。  

In [35]:
print(last_hidden_state.shape)
print(pooler_output.shape)
print(hidden_states[0].shape)

torch.Size([1, 15, 768])
torch.Size([1, 768])
torch.Size([1, 15, 768])


- 数组`[1, 15, 768`表示`[batch_size, sequence_length, hidden_size]`。
- 数组`[1, 768]`表示`[batch_size, hidden_size]`